### End of week 1 exercise
To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [10]:
# constants

MODEL_GPT = "gpt-4o-mini"
# MODEL_OLLAMA = "llama3.2"
MODEL_OLLAMA = "gemma3:270m"

openai_client = OpenAI()

In [3]:
# set up environments

load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    print("No API Key was found.")
elif not openai_api_key.startswith("sk-proj-"):
    print("An API Key was found but it not started with \'sk-proj-\'")
elif openai_api_key.strip() != openai_api_key:
    print("An API Key was found but it contains some space or empty characters at the start or end of the characters.")
else:
    print("An API Key was found and it looks good so far.")

An API Key was found and it looks good so far.


In [4]:
# Setting System Prompt and User Prompt

system_prompt = """
You are an helpful AI assistant that answers concisely.
You are an expert in technical field. 
You have more than 10 years of working experience in technical field.
Whenever a user ask a question you give a detailed answer.
"""

def get_user_prompt(question):

    user_prompt = f"""
Below is the question asked by a user.
Please give a detailed answer for the question asked.

Question:
{question}
"""

    return user_prompt

In [5]:
# Ask a question

question = "We are building a customer support chatbot using a pre-trained LLM (like GPT-4), but it is hallucinating wrong product information and giving inconsistent answers. Walk me through your technical approach to minimize hallucinations and ensure the chatbot only uses our company's official documentation"

In [6]:
# Get gpt-4o-mini to answer, with streaming

def stream_response_using_gpt(question):

    response = openai_client.chat.completions.create(
        model = MODEL_GPT,
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': get_user_prompt(question)}
        ],
        stream= True
    )

    result = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in response:
        result += chunk.choices[0].delta.content or ''
        update_display(Markdown(result), display_id= display_handle.display_id)

In [7]:
stream_response_using_gpt(question)

Building an effective customer support chatbot that leverages a pre-trained LLM like GPT-4 involves addressing the challenge of hallucination—where the model generates incorrect or fabricated information. Here’s a structured technical approach to minimize hallucinations and ensure the chatbot adheres to your company's official documentation.

### 1. **Data Curation and Integration**

**a. Source Official Documentation:**
   - Consolidate all relevant customer support documentation, including user manuals, FAQs, troubleshooting guides, and knowledge base articles. Ensure these documents are up-to-date and comprehensive.

**b. Pre-process the Data:**
   - Clean the data by removing irrelevant sections, standardizing language, and ensuring consistent terminology. Break the documents into manageable chunks for easier retrieval during the chatbot interactions.

### 2. **Fine-Tuning the Model**

**a. Fine-Tuning with Custom Data:**
   - Use your company's official documentation to fine-tune the LLM. This will allow the model to better understand the specific language, context, and formats used in your support materials, reducing the likelihood of hallucinations.

**b. In-context Learning:**
   - Employ in-context learning by providing examples relevant to your products and services when querying the LLM. Structure the input prompts in a way that prioritizes the company’s context.

### 3. **Implement Knowledge Retrieval Mechanisms**

**a. Information Retrieval (IR) System:**
   - Develop an IR system that retrieves relevant snippets from your company's documentation when a user queries the chatbot. This can be done using:
     - **Vector Search:** Index your documents using embedding techniques like Sentence-BERT, and implement semantic search to fetch the most relevant documents based on user queries.
     - **Keyword-based Search:** Use keyword extraction to match user queries with relevant sections of your documentation.

**b. Hybrid Approach:**
   - Combine the LLM’s generative capabilities with the IR system. First, use the IR system to fetch relevant information and then allow the LLM to generate responses based on this retrieved content to ensure accuracy.

### 4. **Prompt Engineering**

**a. Design Effective Prompts:**
   - Craft prompts that steer the model toward specific, fact-based responses. For example, frame queries to the model as tasks or directives like, “Based on the product manual, explain...”, or include constraints that specify only using official documentation.

**b. Contextual Guidance:**
   - Provide meta-prompts that instruct the model to reference specific sections of the documentation during its response process.

### 5. **Validation and Moderation**

**a. Response Validation:**
   - Implement a validation layer that checks the content before it reaches the end user. This can be accomplished using rule-based logic or additional models designed to verify whether the bot's responses align with the official documentation.

**b. Continuous Feedback Loop:**
   - Encourage users to provide feedback on responses. Create a mechanism to flag incorrect information, which can then be reviewed and used to retrain the model or revise the documentation.

### 6. **Monitoring and Maintenance**

**a. Logging and Monitoring:**
   - Use monitoring tools to log interactions, focusing on identifying patterns where the chatbot may hallucinate. Analyze these logs to continuously improve response strategies.

**b. Regular Updates:**
   - Keep your official documentation updated. As the information changes (e.g., new product launches, software updates), ensure these changes are reflected in both the data fed to the model and the retrieval systems.

### 7. **User Experience Design**

**a. Provide Clear Disclaimers:**
   - Make users aware that the chatbot is using a pre-trained model and that responses should be cross-verified with official documentation for critical issues.

**b. Human Escalation:**
   - Maintain a mechanism for users to escalate complex queries to human agents, ensuring that high-stakes or nuanced inquiries receive accurate responses.

By following this structured approach, you can minimize hallucinations in your customer support chatbot and ensure it consistently refers to and utilizes your company’s official documentation to provide accurate information. This enhances user trust and satisfaction while leveraging the capabilities of advanced LLMs.

In [8]:
# Get Llama 3.2 to answer

def stream_response_using_ollama(question):

    OLLAMA_BASE_URL = "http://localhost:11434/v1"

    ollama = OpenAI(base_url= OLLAMA_BASE_URL, api_key= "ollama")

    response = ollama.chat.completions.create(
        model = MODEL_OLLAMA,
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': get_user_prompt(question)}
        ],
        stream= True
    )

    result = ""
    display_handle = display(Markdown(""), display_id= True)

    for chunk in response:
        result += chunk.choices[0].delta.content or ''
        update_display(Markdown(result), display_id= display_handle.display_id)

In [11]:
stream_response_using_ollama(question)

Okay, here's a detailed explanation of my approach to minimizing hallucinations and ensuring the chatbot only uses company-approved documentation:

My technical approach to minimizing hallucinations in a customer support chatbot involves several key steps:

1.  **Data Gathering and Preprocessing:** I will meticulously gather and clean the training data. This includes:
    *   **Removing irrelevant content:** Any user input that doesn't directly relate to the product or functionality.
    *   **Filtering and cleaning:** Removing potentially offensive or misleading content.
    *   **Data Augmentation:**  Using data augmentation techniques to expose more diverse and potentially more accurate information.  This could involve rotating examples, adding new examples, or re-sampling.

2.  **Model Training:** I'll train a large, pre-trained LLM (e.g., GPT-4) on the curated data.  The model will be trained on a large dataset of knowledge and common queries related to the product and its specifications.  The goal is to learn to understand context, generate coherent and contextually relevant responses, and avoid generating incorrect or misleading information.

3.  **Knowledge Retrieval and Generation:** Once the model is trained, I will use its knowledge base to retrieve and generate answers. This process involves:
    *   **Retrieval-Augmented Generation:**  I'll use the model's knowledge base to retrieve information from the documentation and then adapt the responses using the retrieved information. This leverages the model's training data to provide more accurate and reliable answers.
    *   **Prompt Engineering:** I will craft effective prompts that guide the model to generate accurate and contextually relevant answers. The prompt will focus on providing clear instructions, constraints, and examples.

4.  **Refinement and Evaluation:** I will continuously monitor the chatbot's performance and adjust the training data, model architecture, and prompt engineering strategies based on the feedback I receive.  This involves using metrics like:
    *   **Accuracy:** Determining how close the chatbot is to the truth.
    *   **Relevance:** Assessing the relevance of the generated responses to the user's query.
    *   **Fluency:** Evaluating the clarity, coherence, and naturalness of the responses.

5.  **Human-in-the-Loop:**  While I'm primarily focused on automating responses, I will also incorporate a human-in-the-loop system. This means that for frequently asked questions, I'll have human agents available to provide immediate support. This human agent can review the chatbot's responses, identify potential errors, and offer guidance to the chatbot on how to improve its performance.

In summary, my approach is to leverage the strengths of the pre-trained LLM to create a more accurate, efficient, and helpful chatbot. This involves a multi-step process, from data collection and training to model refinement and human oversight. The goal is to provide a consistent and reliable answer to user inquiries, while minimizing the risk of hallucination.
